In [ ]:
"""
Dual-branch semantic segmentation + classification network for crop disease
severity estimation, following a BiSeNetV2-style design:

  - Detail branch: shallow, high-resolution, preserves lesion boundaries
  - Semantic branch: MobileNetV3-Small backbone, coarse contextual features
  - Guided aggregation: bilateral cross-gating fusion of the two branches
  - Segmentation head: pixel-wise severity mask (background/leaf/lesion)
  - Classification head: disease identification, pooled from the semantic
    branch directly (not the fused features -- see notes below)

Designed to be trained on Kaggle (free GPU) and later exported via ONNX for
edge deployment (Raspberry Pi 4, TFLite/ONNX Runtime/ncnn).
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class ConvBNReLU(nn.Module):
    """Standard conv -> batchnorm -> relu block."""

    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=False
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class DetailBranch(nn.Module):
    """
    Shallow, high-resolution branch. 224x224x3 -> 28x28x64 (8x downsample).
    Three conv layers, each halving spatial resolution once (2x, 2x, 2x = 8x).
    """

    def __init__(self, out_ch=64):
        super().__init__()
        self.layer1 = ConvBNReLU(3, 32, stride=2)      # 224 -> 112
        self.layer2 = ConvBNReLU(32, 48, stride=2)     # 112 -> 56
        self.layer3 = ConvBNReLU(48, out_ch, stride=2)  # 56  -> 28

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x  # (B, 64, 28, 28)


class SemanticBranch(nn.Module):
    """
    MobileNetV3-Small backbone. 224x224x3 -> 7x7x576.
    Uses the `.features` module of torchvision's MobileNetV3-Small, which
    ends in a 576-channel feature map before the classifier head.
    """

    def __init__(self, pretrained=True):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        backbone = mobilenet_v3_small(weights=weights)
        self.features = backbone.features  # output: (B, 576, 7, 7) for 224 input

    def forward(self, x):
        return self.features(x)  # (B, 576, 7, 7)


class GuidedAggregation(nn.Module):
    """
    Bilateral guided aggregation layer (BiSeNetV2-style).

    Detail branch features are downsampled to the semantic branch's spatial
    size and gated (multiplied) by a learned attention map from the semantic
    branch; semantic branch features are upsampled to the detail branch's
    spatial size and gated by a learned attention map from the detail branch.
    Both paths are then combined and projected to `out_ch` channels.
    """

    def __init__(self, detail_ch=64, semantic_ch=576, out_ch=128):
        super().__init__()

        # Detail path: downsample to semantic branch's spatial size (28->7)
        self.detail_down = nn.Sequential(
            nn.Conv2d(detail_ch, detail_ch, 3, stride=2, padding=1, groups=detail_ch, bias=False),
            nn.BatchNorm2d(detail_ch),
            nn.Conv2d(detail_ch, detail_ch, 3, stride=2, padding=1, groups=detail_ch, bias=False),
            nn.BatchNorm2d(detail_ch),
        )
        self.semantic_gate_from_detail = nn.Sequential(
            nn.Conv2d(detail_ch, semantic_ch, 1, bias=False),
            nn.Sigmoid(),
        )

        # Semantic path: upsample to detail branch's spatial size (7->28)
        self.semantic_gate_conv = nn.Sequential(
            nn.Conv2d(semantic_ch, detail_ch, 1, bias=False),
            nn.BatchNorm2d(detail_ch),
            nn.Sigmoid(),
        )

        # Final projection after combining both gated paths
        self.project = nn.Sequential(
            nn.Conv2d(detail_ch + semantic_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, detail_feat, semantic_feat):
        # detail_feat: (B, 64, 28, 28), semantic_feat: (B, 576, 7, 7)

        # --- Semantic side gated by (downsampled) detail attention ---
        detail_down = self.detail_down(detail_feat)                    # (B, 64, 7, 7)
        gate_for_semantic = self.semantic_gate_from_detail(detail_down)  # (B, 576, 7, 7)
        semantic_gated = semantic_feat * gate_for_semantic             # (B, 576, 7, 7)
        semantic_up = F.interpolate(
            semantic_gated, size=detail_feat.shape[-2:], mode="bilinear", align_corners=False
        )  # (B, 576, 28, 28)

        # --- Detail side gated by (upsampled) semantic attention ---
        semantic_up_gate = F.interpolate(
            semantic_feat, size=detail_feat.shape[-2:], mode="bilinear", align_corners=False
        )  # (B, 576, 28, 28)
        gate_for_detail = self.semantic_gate_conv(semantic_up_gate)    # (B, 64, 28, 28)
        detail_gated = detail_feat * gate_for_detail                   # (B, 64, 28, 28)

        fused = torch.cat([detail_gated, semantic_up], dim=1)          # (B, 640, 28, 28)
        return self.project(fused)                                     # (B, 128, 28, 28)


class SegmentationHead(nn.Module):
    """
    Predicts class logits at low resolution (28x28), THEN upsamples to
    224x224 -- not the other way around. The final 1x1 class-prediction
    conv is the most expensive layer here when num_classes is large, since
    its cost scales with output_channels x H x W. Running it at 28x28
    instead of 224x224 cuts its cost by (224/28)^2 = 64x, which matters a
    lot on CPU-only edge hardware (Raspberry Pi 4) and matters more the
    more classes you predict.
    """

    def __init__(self, in_ch=128, num_classes=3):
        super().__init__()
        self.conv1 = ConvBNReLU(in_ch, 64)
        self.conv2 = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        x = self.conv1(x)              # (B, 64, 28, 28)
        x = self.conv2(x)              # (B, num_classes, 28, 28) -- cheap, low-res
        x = F.interpolate(
            x, scale_factor=8, mode="bilinear", align_corners=False
        )  # 28 -> 224, just upsampling logits, no extra channel-mixing cost
        return x  # (B, num_classes, 224, 224)


class ClassificationHead(nn.Module):
    """
    Disease classification head. Pools from the semantic branch (7x7x576)
    directly, NOT from the fused guided-aggregation output.

    Rationale: the fused features are shaped by the aggregation layer to
    help segmentation (edge-preserving, gated by the detail branch), which
    isn't necessarily what helps distinguish disease *type*. Pooling
    straight off the backbone keeps this head cheap (GAP + one FC) and
    gives the classifier the full 7x7 receptive field over the whole image,
    rather than funneling it through the detail-focused fusion first.
    """

    def __init__(self, in_ch=576, num_classes=115):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(in_ch, num_classes)

    def forward(self, semantic_feat):
        x = self.pool(semantic_feat).flatten(1)  # (B, 576)
        return self.fc(x)  # (B, num_classes)


class CropDiseaseNet(nn.Module):
    """
    Full model: dual-branch segmentation + disease classification.

    Args:
        num_seg_classes: channels in the segmentation mask
            (e.g. 3 for background/healthy-leaf/lesion)
        num_disease_classes: number of disease categories to classify
        pretrained_backbone: use ImageNet-pretrained MobileNetV3-Small weights
    """

    def __init__(self, num_seg_classes=3, num_disease_classes=115, pretrained_backbone=True):
        super().__init__()
        self.detail_branch = DetailBranch(out_ch=64)
        self.semantic_branch = SemanticBranch(pretrained=pretrained_backbone)
        self.aggregation = GuidedAggregation(detail_ch=64, semantic_ch=576, out_ch=128)
        self.seg_head = SegmentationHead(in_ch=128, num_classes=num_seg_classes)
        self.cls_head = ClassificationHead(in_ch=576, num_classes=num_disease_classes)

    def forward(self, x):
        detail_feat = self.detail_branch(x)        # (B, 64, 28, 28)
        semantic_feat = self.semantic_branch(x)     # (B, 576, 7, 7)
        fused = self.aggregation(detail_feat, semantic_feat)  # (B, 128, 28, 28)

        seg_out = self.seg_head(fused)              # (B, num_seg_classes, 224, 224)
        cls_out = self.cls_head(semantic_feat)       # (B, num_disease_classes)

        return seg_out, cls_out



In [ ]:
# Quick sanity check of tensor shapes end-to-end
model = CropDiseaseNet(num_seg_classes=3, num_disease_classes=115, pretrained_backbone=False)
dummy_input = torch.randn(2, 3, 224, 224)
seg_out, cls_out = model(dummy_input)
print("Segmentation output shape:", seg_out.shape)   # expect (2, 3, 224, 224)
print("Classification output shape:", cls_out.shape)  # expect (2, 115)

num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params:,}")